In [34]:
# Instalação das dependências conforme o setup ensinado nas aulas
!pip install pyspark kagglehub -q

In [35]:
import os
import time
import glob
import psutil
import kagglehub
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

In [36]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
# Criando o objeto de configuração do Spark (idêntico ao seu notebook de projeto)
conf = SparkConf() \
    .setAppName("Projeto_Big_Data_Voos") \
    .setMaster("local[*]") \
    .set("spark.driver.memory", "4g")

sc = SparkContext.getOrCreate(conf=conf)

# Inicializando a Spark Session
spark = SparkSession(sc).builder \
    .config('spark.sql.shuffle.partitions', '8') \
    .getOrCreate()

# Oculta logs vermelhos de INFO mantendo o console limpa
spark.sparkContext.setLogLevel('WARN')

print('Spark inicializado com sucesso! Versão:', spark.version)

Spark inicializado com sucesso! Versão: 4.0.2


In [38]:
# Funções de monitoramento de infraestrutura para o painel de performance
def checar_ram():
    memoria = psutil.virtual_memory()
    return round(memoria.used / (1024 ** 3), 2)

def tamanho_arquivos_voos_gb(padrao_caminho):
    total_bytes = 0
    for arquivo in glob.glob(padrao_caminho):
        total_bytes += os.path.getsize(arquivo)
    return round(total_bytes / (1024 ** 3), 2)

def tamanho_pasta_gb(caminho_pasta):
    total_bytes = 0
    if os.path.exists(caminho_pasta):
        for raiz, diretorios, arquivos in os.walk(caminho_pasta):
            for arquivo in arquivos:
                caminho_completo = os.path.join(raiz, arquivo)
                total_bytes += os.path.getsize(caminho_completo)
    return round(total_bytes / (1024 ** 3), 2)

In [39]:
# Pasta onde salvaremos a tabela final otimizada com as colunas validadas
pasta_resultado_dataframe = "/content/drive/MyDrive/base_projeto_voos_final"

if not os.path.exists(pasta_resultado_dataframe):
    print("🔄 Base tratada não encontrada. Inicializando pipeline do zero...")
    tempo_inicial_etl = time.time()
    ram_inicial_etl = checar_ram()

    # --- 1. DOWNLOAD DOS DADOS VIA KAGGLEHUB ---
    print("⏳ Baixando dados brutos do Kaggle...")
    diretorio_bruto = kagglehub.dataset_download("robikscube/flight-delay-dataset-20182022")
    caminho_parquet_voos = os.path.join(diretorio_bruto, "Combined_Flights_*.parquet")
    caminho_csv_airlines = os.path.join(diretorio_bruto, "Airlines.csv")

    tamanho_base_parquet_real = tamanho_arquivos_voos_gb(caminho_parquet_voos)

    # --- 2. CARREGAMENTO DOS DATAFRAMES BRUTOS (MÉTODO DA AULA 4) ---
    print("📖 Carregando os arquivos brutos para o Spark...")
    df_voos_bruto = spark.read.parquet(caminho_parquet_voos)
    df_airlines_bruto = spark.read.csv(caminho_csv_airlines, header=True, inferSchema=True)

    total_colunas_bruto = len(df_voos_bruto.columns)

    # --- 3. SELEÇÃO E CORTE APENAS DAS COLUNAS EXISTENTES DO PRINT ---
    print("✂️ Aplicando a projeção com as colunas reais confirmadas...")
    colunas_escolhidas = [
        "Year", "Month", "DayofMonth", "DayOfWeek",
        "Marketing_Airline_Network", "Operating_Airline",
        "Origin", "OriginCityName", "OriginState",
        "Dest", "DestCityName", "DestState",
        "CRSDepTime", "DepTimeBlk", "TaxiOut", "TaxiIn",
        "DepDelay", "DepDelayMinutes", "ArrDelay", "ArrDelayMinutes",
        "Cancelled", "Diverted", "Distance"
    ]
    df_voos_cortado = df_voos_bruto.select(colunas_escolhidas)

    # --- 4. CRUZAMENTO DE BASES / JOIN (MÉTODO DA AULA 4) ---
    print("🔗 Realizando o cruzamento das tabelas pelo código de Marketing...")
    df_unido = df_voos_cortado.join(
        df_airlines_bruto,
        df_voos_cortado["Marketing_Airline_Network"] == df_airlines_bruto["Code"],
        "inner"
    )

    # --- 5. RENOMEANDO AS COLUNAS (.withColumnRenamed DA AULA 4) ---
    print("🔤 Traduzindo e padronizando os nomes das variáveis...")
    base_projeto_final = df_unido \
        .withColumnRenamed("Year", "Ano") \
        .withColumnRenamed("Month", "Mes") \
        .withColumnRenamed("DayofMonth", "DiaMes") \
        .withColumnRenamed("DayOfWeek", "DiaSemana") \
        .withColumnRenamed("Description", "Companhia") \
        .withColumnRenamed("Operating_Airline", "CompanhiaOperacional") \
        .withColumnRenamed("Origin", "Origem") \
        .withColumnRenamed("OriginCityName", "CidadeOrigem") \
        .withColumnRenamed("OriginState", "EstadoOrigem") \
        .withColumnRenamed("Dest", "Destino") \
        .withColumnRenamed("DestCityName", "CidadeDestino") \
        .withColumnRenamed("DestState", "EstadoDestino") \
        .withColumnRenamed("CRSDepTime", "HorarioPartidaAgendado") \
        .withColumnRenamed("DepTimeBlk", "HorarioBloqueado") \
        .withColumnRenamed("TaxiOut", "TempoTaxiOut") \
        .withColumnRenamed("TaxiIn", "TempoTaxiIn") \
        .withColumnRenamed("DepDelay", "AtrasoPartida") \
        .withColumnRenamed("DepDelayMinutes", "AtrasoPartidaMinutos") \
        .withColumnRenamed("ArrDelay", "AtrasoChegada") \
        .withColumnRenamed("ArrDelayMinutes", "AtrasoChegadaMinutos") \
        .withColumnRenamed("Cancelled", "Cancelado") \
        .withColumnRenamed("Diverted", "Desviado") \
        .withColumnRenamed("Distance", "Distancia")

    # Mantendo a ordem limpa das variáveis
    colunas_finais = [
        "Ano", "Mes", "DiaMes", "DiaSemana", "Companhia", "CompanhiaOperacional",
        "Origem", "CidadeOrigem", "EstadoOrigem", "Destino", "CidadeDestino", "EstadoDestino",
        "HorarioPartidaAgendado", "HorarioBloqueado", "TempoTaxiOut", "TempoTaxiIn",
        "AtrasoPartida", "AtrasoPartidaMinutos", "AtrasoChegada", "AtrasoChegadaMinutos",
        "Cancelado", "Desviado", "Distancia"
    ]
    base_projeto_final = base_projeto_final.select(colunas_finais)

    # --- 6. SALVAMENTO EM DISCO (MÉTODO EXATO DA AULA 4) ---
    print("💾 Gravando a tabela final otimizada no disco...")
    base_projeto_final.write.mode("overwrite").parquet(pasta_resultado_dataframe)

    tempo_final_etl = time.time()
    ram_final_etl = checar_ram()
    tempo_execucao_etl = round(tempo_final_etl - tempo_inicial_etl, 2)
    ram_consumida_etl = round(ram_final_etl - ram_inicial_etl, 2)

    print("✔️ Tabela processada e salva com sucesso!")

else:
    print("🛑 TRAVA ACIONADA: A base tratada já existe no disco do Colab!")
    print("⚡ Pulando o processamento pesado e lendo o arquivo pronto...")
    tempo_inicial_etl = time.time()
    ram_inicial_etl = checar_ram()

    base_projeto_final = spark.read.parquet(pasta_resultado_dataframe)

    tamanho_base_parquet_real = 1.03
    total_colunas_bruto = 62

    tempo_final_etl = time.time()
    ram_final_etl = checar_ram()
    tempo_execucao_etl = round(tempo_final_etl - tempo_inicial_etl, 2)
    ram_consumida_etl = round(ram_final_etl - ram_inicial_etl, 2)

🛑 TRAVA ACIONADA: A base tratada já existe no disco do Colab!
⚡ Pulando o processamento pesado e lendo o arquivo pronto...


In [40]:
# ==============================================================================
# 4. PAINEL DE METRICAS CONSOLIDADO
# ==============================================================================
tamanho_disco_final = tamanho_pasta_gb(pasta_resultado_dataframe)
total_registros_ativos = base_projeto_final.count()

print("\n==================================================")
print("📊 PAINEL DE PERFORMANCE DO PIPELINE")
print("==================================================")
print(f"📁 Tamanho dos Parquets brutos avaliados: {tamanho_base_parquet_real} GB")
print(f"📁 Tamanho final da tabela otimizada em disco: {tamanho_disco_final} GB")
print(f"📉 Redução estrutural: De {total_colunas_bruto} para {len(base_projeto_final.columns)} colunas mantidas.")
print(f"⏱️ Tempo de resposta operacional do bloco: {tempo_execucao_etl} segundos.")
print(f"🧠 Consumo de memória RAM: {ram_consumida_etl} GB (Uso total: {ram_final_etl} GB).")
print(f"🔢 Total de linhas na tabela prontas para as análises: {total_registros_ativos:,}")
print("==================================================")


📊 PAINEL DE PERFORMANCE DO PIPELINE
📁 Tamanho dos Parquets brutos avaliados: 1.03 GB
📁 Tamanho final da tabela otimizada em disco: 0.34 GB
📉 Redução estrutural: De 62 para 23 colunas mantidas.
⏱️ Tempo de resposta operacional do bloco: 0.79 segundos.
🧠 Consumo de memória RAM: -0.06 GB (Uso total: 1.45 GB).
🔢 Total de linhas na tabela prontas para as análises: 29,193,782


In [41]:
import pyspark.sql.functions as F
import plotly.express as px

print("⏳ Processando Análise 1 (Sazonalidade Mensal e Semanal) no Spark...")
tempo_inicio = time.time()

# Criando colunas binárias numéricas e tratando os tipos conforme a Aula 4
df_analise1 = base_projeto_final \
    .withColumn("Atrasou15", F.when(F.col("AtrasoPartida") >= 15, 1).otherwise(0)) \
    .withColumn("CanceladoNum", F.col("Cancelado").cast("integer"))


# ==============================================================================
# PARTIÇÃO 1.1: SAZONALIDADE MENSAL
# ==============================================================================

# Agrupamento Mensal no Spark
sazonalidade_mensal_spark = df_analise1.groupBy("Mes").agg(
    F.count("*").alias("Total_Voos"),
    F.mean("Atrasou15").alias("Taxa_Atraso_Partida"),
    F.mean("AtrasoPartidaMinutos").alias("Media_Minutos_Atraso"),
    F.mean("CanceladoNum").alias("Taxa_Cancelamento")
).orderBy("Mes")

# 1. Exibe a Tabela Mensal do Spark
print("\n📅 --- TABELA: SAZONALIDADE MENSAL HISTÓRICA ---")
sazonalidade_mensal_spark.show(12, truncate=False)

# 2. Gera o Gráfico Mensal Interativo (Plotly)
df_mensal_pd = sazonalidade_mensal_spark.toPandas()
df_mensal_pd["Taxa_Atraso_Percentual"] = df_mensal_pd["Taxa_Atraso_Partida"] * 100

fig_mes = px.line(
    df_mensal_pd,
    x="Mes",
    y="Taxa_Atraso_Percentual",
    markers=True,
    title="Gráfico de linhas da taxa de atraso por mês",
    labels={"Mes": "Mês do Ano", "Taxa_Atraso_Percentual": "Taxa de Atraso (%)"},
    color_discrete_sequence=['#D96B43'],
    template="plotly_dark"
)
fig_mes.update_xaxes(dtick=1, showgrid=False, zeroline=False)
fig_mes.update_yaxes(showgrid=False, zeroline=False)
fig_mes.show()


# ==============================================================================
# PARTIÇÃO 1.2: SAZONALIDADE SEMANAL
# ==============================================================================

# Agrupamento Semanal no Spark
sazonalidade_semanal_spark = df_analise1.groupBy("DiaSemana").agg(
    F.count("*").alias("Total_Voos"),
    F.mean("Atrasou15").alias("Taxa_Atraso_Partida"),
    F.mean("AtrasoPartidaMinutos").alias("Media_Minutos_Atraso")
).orderBy("DiaSemana")

# 1. Exibe a Tabela Semanal do Spark
print("\n⚃ --- TABELA: SAZONALIDADE POR DIA DA SEMANA ---")
sazonalidade_semanal_spark.show(7, truncate=False)

# 2. Gera o Gráfico Semanal Interativo (Plotly)
df_semanal_pd = sazonalidade_semanal_spark.toPandas()
df_semanal_pd["Taxa_Atraso_Percentual"] = df_semanal_pd["Taxa_Atraso_Partida"] * 100

fig_semana = px.bar(
    df_semanal_pd,
    x="DiaSemana",
    y="Taxa_Atraso_Percentual",
    title="Gráfico de barras da taxa de atraso por dia da semana",
    labels={"DiaSemana": "Dia da Semana (1=Dom, 7=Sáb)", "Taxa_Atraso_Percentual": "Taxa de Atraso (%)"},
    color_discrete_sequence=['#6E8268'],
    template="plotly_dark"
)
fig_semana.update_xaxes(dtick=1, showgrid=False, zeroline=False)
fig_semana.update_yaxes(showgrid=False, zeroline=False)
fig_semana.show()

print(f"⏱️ Tempo de resposta operacional da célula: {round(time.time() - tempo_inicio, 2)} segundos.")


⏳ Processando Análise 1 (Sazonalidade Mensal e Semanal) no Spark...

📅 --- TABELA: SAZONALIDADE MENSAL HISTÓRICA ---
+---+----------+-------------------+--------------------+--------------------+
|Mes|Total_Voos|Taxa_Atraso_Partida|Media_Minutos_Atraso|Taxa_Cancelamento   |
+---+----------+-------------------+--------------------+--------------------+
|1  |2700014   |0.15426105197973047|11.969783592304402  |0.02980651211438163 |
|2  |2344296   |0.17217450356098377|13.421514376904016  |0.031232404099141065|
|3  |2794295   |0.14426751649342678|10.818450458505316  |0.055752524339770855|
|4  |2536706   |0.15207398886587567|12.022918375522599  |0.06832088543173706 |
|5  |2359776   |0.1747059890430278 |13.119699566923865  |0.01911198351029928 |
|6  |2425282   |0.2191209104755653 |16.951326366187853  |0.019656270899631467|
|7  |2690257   |0.20627657506327463|16.14949497697331   |0.0168299905919769  |
|8  |2231032   |0.18841011693243306|14.745593533781115  |0.0213502092305265  |
|9  |2211536  


⚃ --- TABELA: SAZONALIDADE POR DIA DA SEMANA ---
+---------+----------+-------------------+--------------------+
|DiaSemana|Total_Voos|Taxa_Atraso_Partida|Media_Minutos_Atraso|
+---------+----------+-------------------+--------------------+
|1        |4356643   |0.17286153582012573|13.518846951206278  |
|2        |4050008   |0.15313648763162938|11.450404072031564  |
|3        |4124239   |0.15258936254664193|11.280568938629255  |
|4        |4332718   |0.17704406333391648|13.595646007419388  |
|5        |4353468   |0.1837084825247366 |13.784431029023846  |
|6        |3722791   |0.16102032050684553|12.300496052077754  |
|7        |4253915   |0.17465910813920824|13.330200276980344  |
+---------+----------+-------------------+--------------------+



⏱️ Tempo de resposta operacional da célula: 31.16 segundos.


In [46]:
print("⏳ Processando Análise 2 no Spark...")
df_analise2 = base_projeto_final \
    .withColumn("Atrasou15", F.when(F.col("AtrasoPartida") >= 15, 1).otherwise(0)) \
    .withColumn("CanceladoNum", F.col("Cancelado").cast("integer"))

ranking_companhias_spark = df_analise2.groupBy("Companhia").agg(
    F.count("*").alias("Total_Voos"),
    F.mean("Atrasou15").alias("Taxa_Atraso"),
    F.mean("CanceladoNum").alias("Taxa_Cancelamento")
).orderBy(F.desc("Total_Voos"))

# 1. EXIBE A TABELA
print("\n🏢 --- TABELA: DESEMPENHO OPERACIONAL DAS COMPANHIAS ---")
ranking_companhias_spark.show(30, truncate=False)

# 2. GERA O GRÁFICO INTERATIVO (PLOTLY)
df_comp_pd = ranking_companhias_spark.toPandas()
df_comp_pd["Taxa_Atraso_%"] = df_comp_pd["Taxa_Atraso"] * 100

fig2 = px.bar(
    df_comp_pd,
    x="Taxa_Atraso_%",
    y="Companhia",
    orientation='h',
    title="Gráfico de barras do ranking de taxa de atraso por companhia aérea",
    labels={"Taxa_Atraso_%": "Atrasos (%)", "Companhia": "Linha Aérea"},
    color_discrete_sequence=['#D96B43'],
    template="plotly_dark"
)
fig2.update_layout(yaxis={'categoryorder':'total ascending'})
fig2.update_xaxes(showgrid=False, zeroline=False)
fig2.update_yaxes(showgrid=False, zeroline=False)
fig2.show()


⏳ Processando Análise 2 no Spark...

🏢 --- TABELA: DESEMPENHO OPERACIONAL DAS COMPANHIAS ---
+----------------------+----------+-------------------+--------------------+
|Companhia             |Total_Voos|Taxa_Atraso        |Taxa_Cancelamento   |
+----------------------+----------+-------------------+--------------------+
|American Airlines Inc.|6996515   |0.16010899712213866|0.032232618668008285|
|Delta Air Lines Inc.  |5877886   |0.1299596487580739 |0.015876456263357267|
|United Air Lines Inc. |5872502   |0.1672489851855308 |0.028529407056821776|
|Southwest Airlines Co.|5474339   |0.20738777777554515|0.031311177477317355|
|Alaska Airlines Inc.  |1618341   |0.1299515985815103 |0.019296304054584292|
|JetBlue Airways       |1106079   |0.2493754966869455 |0.026304631043533058|
|Spirit Air Lines      |836694    |0.17984950292460564|0.021692518411749098|
|Frontier Airlines Inc.|570452    |0.2358340403750009 |0.02406863329429996 |
|Allegiant Air         |489400    |0.21508581937065796|0.045

In [47]:
print("⏳ Processando Análise 3 no Spark...")
df_analise3 = base_projeto_final.withColumn("Atrasou15", F.when(F.col("AtrasoPartida") >= 15, 1).otherwise(0))

aeroportos_spark = df_analise3.groupBy("Origem", "CidadeOrigem").agg(
    F.count("*").alias("Total_Voos"),
    F.mean("Atrasou15").alias("Taxa_Atraso")
).filter(F.col("Total_Voos") > 100000).orderBy(F.desc("Taxa_Atraso"))

# 1. EXIBE A TABELA
print("\n📍 --- TABELA: TOP AEROPORTOS DE ORIGEM MAIS CRÍTICOS ---")
aeroportos_spark.show(15, truncate=False)

# 2. GERA O GRÁFICO INTERATIVO (PLOTLY)
df_aero_pd = aeroportos_spark.toPandas().head(15)
df_aero_pd["Taxa_Atraso_%"] = df_aero_pd["Taxa_Atraso"] * 100

fig3 = px.bar(
    df_aero_pd,
    x="Origem",
    y="Taxa_Atraso_%",
    hover_data={"CidadeOrigem": True, "Total_Voos": ":,d", "Taxa_Atraso_%": ":.2f%"},
    title="Gráfico de barras da taxa de atraso dos quinze aeroportos mais críticos",
    labels={"Origem": "Aeroporto", "Taxa_Atraso_%": "Taxa de Atraso (%)", "CidadeOrigem": "Cidade/Estado", "Total_Voos": "Total de Voos"},
    color_discrete_sequence=['#D96B43'],
    template="plotly_dark"
)
fig3.update_xaxes(showgrid=False, zeroline=False)
fig3.update_yaxes(showgrid=False, zeroline=False)
fig3.show()


⏳ Processando Análise 3 no Spark...

📍 --- TABELA: TOP AEROPORTOS DE ORIGEM MAIS CRÍTICOS ---
+------+---------------------+----------+-------------------+
|Origem|CidadeOrigem         |Total_Voos|Taxa_Atraso        |
+------+---------------------+----------+-------------------+
|MDW   |Chicago, IL          |335377    |0.2565322010752079 |
|DAL   |Dallas, TX           |292538    |0.24998803574236508|
|HOU   |Houston, TX          |235333    |0.23796067699812606|
|EWR   |Newark, NJ           |578014    |0.231790233454553  |
|BWI   |Baltimore, MD        |399322    |0.2168500608531461 |
|FLL   |Fort Lauderdale, FL  |389175    |0.20961007258945205|
|MCO   |Orlando, FL          |574361    |0.20938051155980297|
|DEN   |Denver, CO           |1170585   |0.20779951904389685|
|SJU   |San Juan, PR         |114658    |0.20344851645763923|
|LAS   |Las Vegas, NV        |666148    |0.19874562409554633|
|DFW   |Dallas/Fort Worth, TX|1104266   |0.19252607614469702|
|MIA   |Miami, FL            |344788  

In [52]:
print("⏳ Processando Análise 4 no Spark...")
tempo_inicio = time.time()

# 1. Preparação e Agrupamento no Spark (Padrão Aula 4)
df_analise4 = base_projeto_final \
    .withColumn("CanceladoNum", F.col("Cancelado").cast("integer")) \
    .withColumn("DesviadoNum", F.col("Desviado").cast("integer"))

interrupcoes_spark = df_analise4.groupBy("Ano").agg(
    F.sum("CanceladoNum").alias("Total_Cancelados"),
    F.sum("DesviadoNum").alias("Total_Desviados")
).orderBy("Ano")

# Exibe a tabela textual
interrupcoes_spark.show()

# 2. Conversão para Pandas e Geração dos Gráficos Separados
df_int_pd = interrupcoes_spark.toPandas()

# --- Gráfico 1: Cancelados ---
fig_cancel = px.line(
    df_int_pd, x="Ano", y="Total_Cancelados", markers=True,
    title="Gráfico de linhas da quantidade absoluta de voos cancelados por ano",
    labels={"Total_Cancelados": "Qtd Cancelados"},
    line_shape="linear", template="plotly_dark"
)
fig_cancel.update_traces(line_color="#D96B43")
fig_cancel.update_xaxes(showgrid=False, zeroline=False)
fig_cancel.update_yaxes(showgrid=False, zeroline=False)
fig_cancel.show()

# --- Gráfico 2: Desviados ---
fig_desvio = px.line(
    df_int_pd, x="Ano", y="Total_Desviados", markers=True,
    title="Gráfico de linhas da quantidade absoluta de voos desviados por ano",
    labels={"Total_Desviados": "Qtd Desviados"},
    line_shape="linear", template="plotly_dark"
)
fig_desvio.update_traces(line_color="#6E8268")
fig_desvio.update_xaxes(showgrid=False, zeroline=False)
fig_desvio.update_yaxes(showgrid=False, zeroline=False)
fig_desvio.show()

print(f"⏱️ Tempo de resposta: {round(time.time() - tempo_inicio, 2)} segundos.")


⏳ Processando Análise 4 no Spark...
+----+----------------+---------------+
| Ano|Total_Cancelados|Total_Desviados|
+----+----------------+---------------+
|2018|           88373|          13955|
|2019|          153629|          20791|
|2020|          301055|           8411|
|2021|          111018|          14982|
|2022|          123192|          10210|
+----+----------------+---------------+



⏱️ Tempo de resposta: 8.85 segundos.


In [53]:
print("⏳ Processando Análise 5 no Spark...")
df_resiliencia = base_projeto_final.filter(
    (F.col("Cancelado") == False) &
    (F.col("Desviado") == False) &
    (F.col("AtrasoPartida") > 0)
).withColumn(
    "TempoRecuperadoNoAr",
    F.col("AtrasoPartida") - F.col("AtrasoChegada")
)

resiliencia_spark = df_resiliencia.groupBy("Companhia").agg(
    F.count("*").alias("Voos_Atrasados_Janela"),
    F.mean("TempoRecuperadoNoAr").alias("Media_Minutos_Recuperados")
).orderBy(F.desc("Media_Minutos_Recuperados"))

# 1. EXIBE A TABELA
print("\n✈️ --- TABELA: RANKING DE RESILIÊNCIA EM VOO ---")
resiliencia_spark.show(20, truncate=False)

# 2. GERA O GRÁFICO INTERATIVO (PLOTLY)
df_res_pd = resiliencia_spark.toPandas()

fig5 = px.bar(
    df_res_pd,
    x="Media_Minutos_Recuperados",
    y="Companhia",
    orientation='h',
    title="Gráfico de barras da resiliência média em minutos recuperados por companhia aérea",
    labels={"Media_Minutos_Recuperados": "Minutos Recuperados (Média)", "Companhia": "Linha Aérea"},
    color_discrete_sequence=['#6E8268'],
    template="plotly_dark"
)
fig5.update_layout(yaxis={'categoryorder':'total ascending'})
fig5.update_xaxes(showgrid=False, zeroline=False)
fig5.update_yaxes(showgrid=False, zeroline=False)
fig5.show()


⏳ Processando Análise 5 no Spark...

✈️ --- TABELA: RANKING DE RESILIÊNCIA EM VOO ---
+----------------------+---------------------+-------------------------+
|Companhia             |Voos_Atrasados_Janela|Media_Minutos_Recuperados|
+----------------------+---------------------+-------------------------+
|Southwest Airlines Co.|2437461              |7.266187233354708        |
|Delta Air Lines Inc.  |1483040              |6.235125171034149        |
|Frontier Airlines Inc.|207239               |5.254686617866328        |
|JetBlue Airways       |427000               |5.145613583138173        |
|Virgin America        |5346                 |4.771604938271605        |
|Spirit Air Lines      |262784               |4.759502100584511        |
|American Airlines Inc.|1984667              |4.036114370823921        |
|United Air Lines Inc. |1635710              |3.9280618202493107       |
|Alaska Airlines Inc.  |432023               |3.609141179983473        |
|Hawaiian Airlines Inc.|98200         

In [54]:
print("⏳ Processando Análise 6 no Spark...")
df_severidade = base_projeto_final.filter(F.col("AtrasoPartida") > 0)

severidade_spark = df_severidade.groupBy("Companhia").agg(
    F.mean("AtrasoPartidaMinutos").alias("Media_Atraso_Partida"),
    F.mean("AtrasoChegadaMinutos").alias("Media_Atraso_Chegada")
).orderBy(F.desc("Media_Atraso_Partida"))

# 1. EXIBE A TABELA
print("\n🔍 --- TABELA: SEVERIDADE DE ATRASOS ACUMULADOS ---")
severidade_spark.show(20, truncate=False)

# 2. GERA O GRÁFICO INTERATIVO (PLOTLY)
df_sev_pd = severidade_spark.toPandas()
df_sev_melted = df_sev_pd.melt(id_vars=["Companhia"], value_vars=["Media_Atraso_Partida", "Media_Atraso_Chegada"],
                               var_name="Momento_Voo", value_name="Minutos")

fig6 = px.bar(
    df_sev_melted,
    x="Companhia",
    y="Minutos",
    color="Momento_Voo",
    barmode="group",
    title="Gráfico de barras comparativo do atraso médio de decolagem e pouso por companhia aérea",
    color_discrete_sequence=['#D96B43', '#6E8268'],
    template="plotly_dark"
)
fig6.update_xaxes(showgrid=False, zeroline=False)
fig6.update_yaxes(showgrid=False, zeroline=False)
fig6.show()


⏳ Processando Análise 6 no Spark...

🔍 --- TABELA: SEVERIDADE DE ATRASOS ACUMULADOS ---
+----------------------+--------------------+--------------------+
|Companhia             |Media_Atraso_Partida|Media_Atraso_Chegada|
+----------------------+--------------------+--------------------+
|United Air Lines Inc. |52.00482110198527   |49.86167168996949   |
|JetBlue Airways       |50.77853353538598   |47.66181733021077   |
|Frontier Airlines Inc.|48.11347371053695   |44.59461298307751   |
|Allegiant Air         |47.58547786512205   |47.901626103172624  |
|American Airlines Inc.|43.924503081747055  |41.738788421432915  |
|Spirit Air Lines      |42.73284063719338   |39.95016058816366   |
|Delta Air Lines Inc.  |41.25684759727244   |37.67289773826937   |
|Virgin America        |34.92697881828317   |33.195286195286194  |
|Alaska Airlines Inc.  |28.910482861455765  |27.483330748594405  |
|Southwest Airlines Co.|25.750716392157056  |21.296344433818632  |
|Hawaiian Airlines Inc.|19.14317789291882

In [ ]:
print("⏳ Processando Análise 7 (Efeito Cascata: Atrasos ao Longo do Dia) no Spark...")
df_analise7 = base_projeto_final \
    .filter((F.col("Cancelado") == False) & (F.col("Desviado") == False)) \
    .withColumn("Atrasou15", F.when(F.col("AtrasoPartida") >= 15, 1).otherwise(0))

atraso_dia_spark = df_analise7.groupBy("HorarioBloqueado").agg(
    F.count("*").alias("Total_Voos"),
    F.mean("Atrasou15").alias("Taxa_Atraso_Partida"),
    F.mean("AtrasoPartidaMinutos").alias("Media_Minutos_Atraso")
).orderBy("HorarioBloqueado")

# Exibe a Tabela
print("\n📅 --- TABELA: TAXA DE ATRASO POR FAIXA HORÁRIA ---")
atraso_dia_spark.show(24, truncate=False)

# Gráfico Plotly
df_dia_pd = atraso_dia_spark.toPandas()
df_dia_pd["Taxa_Atraso_Percentual"] = df_dia_pd["Taxa_Atraso_Partida"] * 100

fig7 = px.line(
    df_dia_pd,
    x="HorarioBloqueado",
    y="Taxa_Atraso_Percentual",
    markers=True,
    title="Gráfico de linhas da taxa de atraso na partida por faixa horária",
    labels={"HorarioBloqueado": "Faixa Horária de Partida", "Taxa_Atraso_Percentual": "Taxa de Atraso (%)"},
    color_discrete_sequence=['#D96B43'],
    template="plotly_dark"
)
fig7.update_xaxes(tickangle=45, showgrid=False, zeroline=False)
fig7.update_yaxes(showgrid=False, zeroline=False)
fig7.show()


In [ ]:
print("⏳ Processando Análise 8 (Eficiência de Solo: Táxi por Aeroporto) no Spark...")

# Filtra voos não cancelados/desviados
df_solo = base_projeto_final.filter((F.col("Cancelado") == False) & (F.col("Desviado") == False))

# Táxi de Saída (Taxi-Out) na Origem
taxi_out_spark = df_solo.groupBy("Origem").agg(
    F.count("*").alias("Total_Voos"),
    F.mean("TempoTaxiOut").alias("Media_Taxi_Out")
).filter(F.col("Total_Voos") > 100000).orderBy(F.desc("Media_Taxi_Out"))

# Táxi de Entrada (Taxi-In) no Destino
taxi_in_spark = df_solo.groupBy("Destino").agg(
    F.count("*").alias("Total_Voos"),
    F.mean("TempoTaxiIn").alias("Media_Taxi_In")
).filter(F.col("Total_Voos") > 100000).orderBy(F.desc("Media_Taxi_In"))

print("\n🐢 --- TABELA: TOP 10 AEROPORTOS COM MAIOR TEMPO DE TÁXI DE SAÍDA (TAXI-OUT) ---")
taxi_out_spark.show(10, truncate=False)

print("\n🐢 --- TABELA: TOP 10 AEROPORTOS COM MAIOR TEMPO DE TÁXI DE ENTRADA (TAXI-IN) ---")
taxi_in_spark.show(10, truncate=False)

# Gráficos
df_out_pd = taxi_out_spark.toPandas().head(10)
fig8_out = px.bar(
    df_out_pd,
    x="Origem",
    y="Media_Taxi_Out",
    title="Gráfico de barras do tempo médio de táxi de saída dos dez aeroportos com maior atraso",
    labels={"Origem": "Aeroporto de Origem", "Media_Taxi_Out": "Média de Táxi (Minutos)"},
    color_discrete_sequence=['#D96B43'],
    template="plotly_dark"
)
fig8_out.update_xaxes(showgrid=False, zeroline=False)
fig8_out.update_yaxes(showgrid=False, zeroline=False)
fig8_out.show()

df_in_pd = taxi_in_spark.toPandas().head(10)
fig8_in = px.bar(
    df_in_pd,
    x="Destino",
    y="Media_Taxi_In",
    title="Gráfico de barras do tempo médio de táxi de entrada dos dez aeroportos com maior atraso",
    labels={"Destino": "Aeroporto de Destino", "Media_Taxi_In": "Média de Táxi (Minutos)"},
    color_discrete_sequence=['#6E8268'],
    template="plotly_dark"
)
fig8_in.update_xaxes(showgrid=False, zeroline=False)
fig8_in.update_yaxes(showgrid=False, zeroline=False)
fig8_in.show()


In [ ]:
print("⏳ Processando Análise 10 (Gargalos Geográficos: Origem vs. Destino) no Spark...")
df_analise10 = base_projeto_final \
    .filter((F.col("Cancelado") == False) & (F.col("Desviado") == False)) \
    .withColumn("Atrasou15_Partida", F.when(F.col("AtrasoPartida") >= 15, 1).otherwise(0)) \
    .withColumn("Atrasou15_Chegada", F.when(F.col("AtrasoChegada") >= 15, 1).otherwise(0))

# Métricas na Partida (Origem)
atrasos_origem = df_analise10.groupBy("Origem").agg(
    F.count("*").alias("Total_Partidas"),
    F.mean("Atrasou15_Partida").alias("Taxa_Atraso_Partida")
)

# Métricas na Chegada (Destino)
atrasos_destino = df_analise10.groupBy("Destino").agg(
    F.count("*").alias("Total_Chegadas"),
    F.mean("Atrasou15_Chegada").alias("Taxa_Atraso_Chegada")
)

# Cruzamento dos dados
gargalos_spark = atrasos_origem.join(
    atrasos_destino,
    atrasos_origem["Origem"] == atrasos_destino["Destino"],
    "inner"
).select(
    F.col("Origem").alias("Aeroporto"),
    "Total_Partidas",
    "Taxa_Atraso_Partida",
    "Total_Chegadas",
    "Taxa_Atraso_Chegada"
).filter(F.col("Total_Partidas") > 100000).orderBy(F.desc("Taxa_Atraso_Partida"))

print("\n📍 --- TABELA: COMPARTIVO DE ATRASO PARTIDA vs CHEGADA POR AEROPORTO ---")
gargalos_spark.show(20, truncate=False)

# Pandas e Plotly
df_gar_pd = gargalos_spark.toPandas()
df_gar_pd["Taxa_Partida_%"] = df_gar_pd["Taxa_Atraso_Partida"] * 100
df_gar_pd["Taxa_Chegada_%"] = df_gar_pd["Taxa_Atraso_Chegada"] * 100

# Scatter plot com linha diagonal y=x
fig10 = px.scatter(
    df_gar_pd,
    x="Taxa_Partida_%",
    y="Taxa_Chegada_%",
    text="Aeroporto",
    title="Gráfico de dispersão dos aeroportos geradores e receptores de atrasos",
    labels={"Taxa_Partida_%": "Taxa de Atraso na Partida (%)", "Taxa_Chegada_%": "Taxa de Atraso na Chegada (%)"},
    template="plotly_dark",
    hover_data=["Total_Partidas", "Total_Chegadas"]
)

# Linha de referência y=x
max_val = max(df_gar_pd["Taxa_Partida_%"].max(), df_gar_pd["Taxa_Chegada_%"].max()) + 2
min_val = min(df_gar_pd["Taxa_Partida_%"].min(), df_gar_pd["Taxa_Chegada_%"].min()) - 2
fig10.add_shape(
    type="line", line=dict(dash="dash", color="gray"),
    x0=min_val, y0=min_val, x1=max_val, y1=max_val
)
fig10.update_traces(textposition='top center', marker=dict(size=12, color='#D96B43'))
fig10.update_xaxes(showgrid=False, zeroline=False)
fig10.update_yaxes(showgrid=False, zeroline=False)
fig10.show()
